In [1]:
!pip3 install --upgrade webull-openapi-python-sdk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 7.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 8.7 MB/s eta 0:00:00a 0:00:01
  Created wheel for paho-mqtt: filename=paho_mqtt-1.6.1-py3-none-any.whl size=62117 sha256=af16d2dfe88499d3eaf27d7058c5f62403a28f02d9d57421393ac5e9649385c1
  Stored in directory: /Users/yela/Library/Caches/pip/wheels/23/d5/af/1f3cbcc350dec9d8e95040f388e0163d132eff0c9a453db659
Successfully built paho-mqtt
  Attempting uninstall: jmespath
    Found existing installation: jmespath 1.0.1
    Uninstalling jmespath-1.0.1:
      Successfully uninstalled jmespath-1.0.1
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.0
    Uninstalling cryptography-43.0.0:
      Successfully uninstalled cryptography-43.0.0


In [3]:
http_api = 'api.webull.com'
test_account_id = 'J6HA4EBQRQFJD2J6NQH0F7M649'
app_key = 'a88f2efed4dca02b9bc1a3cecbc35dba'
app_secret = 'c2895b3526cc7c7588758351ddf425d6'

In [6]:
import json
from webull.core.client import ApiClient
from webull.trade.trade_client import TradeClient

api_client = ApiClient(app_key, app_secret, "us")
api_client.add_endpoint("us", "us-openapi-alb.uat.webullbroker.com")

trade_client = TradeClient(api_client)
res = trade_client.account_v2.get_account_list()
if res.status_code == 200:
    print("Success!", json.dumps(res.json(), indent=2))
else:
    print("Error:", res.status_code, res.text)

8462151936 2026-05-13 19:54:50,629 webull.core.http.initializer.client_initializer INFO _check_token_enable result is True
8462151936 2026-05-13 19:54:50,629 webull.core.http.initializer.client_initializer INFO _check_token_enable result is True
8462151936 2026-05-13 19:54:50,629 webull.core.http.initializer.client_initializer INFO _check_token_enable result is True
8462151936 2026-05-13 19:54:50,633 webull.core.http.initializer.token.token_storage INFO storage_token uses the default configuration, conf.
8462151936 2026-05-13 19:54:50,633 webull.core.http.initializer.token.token_storage INFO storage_token uses the default configuration, conf.
8462151936 2026-05-13 19:54:50,633 webull.core.http.initializer.token.token_storage INFO storage_token uses the default configuration, conf.
8462151936 2026-05-13 19:54:50,638 webull.core.http.initializer.token.token_storage INFO storage_token initialized path:/Users/yela/Documents/University Programming/RL_quantphemes_2/notebooks/conf/token.txt.


KeyboardInterrupt: 

In [8]:
import pandas as pd
from datetime import datetime, timedelta, timezone
import logging

log = logging.getLogger(__name__)
log.setLevel(logging.INFO)

# Use SPY as S&P 500 proxy (Webull generally exposes tradable symbols, not index history directly)
symbol = "SPY"
interval = "5m"
years = 10

# Try to import QuoteClient from known locations in the webull SDK.
QuoteClient = None
try:
    from webull.quote.quote_client import QuoteClient  # newer layout
except Exception:
    try:
        from webull.quote_client import QuoteClient  # older layout
    except Exception:
        QuoteClient = None

all_rows: list[dict] = []

if QuoteClient is not None:
    try:
        quote_client = QuoteClient(api_client)

        end_dt = datetime.now(timezone.utc)
        start_dt = end_dt - timedelta(days=365 * years)

        # Webull APIs usually return limited bars per call, so fetch in chunks.
        limit_per_call = 1000
        cursor_end = end_dt

        while cursor_end > start_dt:
            # handle a couple of SDK shapes: either quote_client.kline.get_kline or quote_client.get_kline
            if hasattr(quote_client, "kline") and callable(getattr(quote_client.kline, "get_kline", None)):
                res = quote_client.kline.get_kline(
                    symbol=symbol,
                    interval=interval,
                    end_time=int(cursor_end.timestamp() * 1000),
                    count=limit_per_call,
                )
            elif callable(getattr(quote_client, "get_kline", None)):
                res = quote_client.get_kline(
                    symbol=symbol,
                    interval=interval,
                    end_time=int(cursor_end.timestamp() * 1000),
                    count=limit_per_call,
                )
            else:
                raise RuntimeError("Installed webull SDK does not expose a kline/get_kline API.")

            # normalize response payload (SDKs differ)
            if hasattr(res, "status_code"):
                if res.status_code != 200:
                    raise RuntimeError(f"Webull API error: {res.status_code} {getattr(res, 'text', '')}")
                payload = res.json()
            elif isinstance(res, dict):
                payload = res
            else:
                # try to call json if present
                payload = res.json() if hasattr(res, "json") else res

            # Try common keys for bars
            if isinstance(payload, dict):
                for key in ("data", "items", "klines", "candles", "list"):
                    if key in payload and isinstance(payload[key], (list, tuple)):
                        bars = payload[key]
                        break
                else:
                    # fallback: maybe payload itself is a list
                    bars = payload if isinstance(payload, list) else []
            elif isinstance(payload, list):
                bars = payload
            else:
                bars = []

            if not bars:
                break

            all_rows.extend(bars)

            # Move cursor to just before oldest bar in this batch
            # try common timestamp field names
            ts_candidates = []
            for b in bars:
                for ts_key in ("timestamp", "ts", "time", "openTime"):
                    if ts_key in b:
                        try:
                            ts_candidates.append(int(b[ts_key]))
                        except Exception:
                            pass
                        break
            if not ts_candidates:
                break

            oldest_ts_ms = min(ts_candidates)
            cursor_end = datetime.fromtimestamp((oldest_ts_ms - 1) / 1000, tz=timezone.utc)

            if oldest_ts_ms <= int(start_dt.timestamp() * 1000):
                break

    except Exception as e:
        log.warning("Webull QuoteClient path failed, falling back to yfinance: %s", e)
        QuoteClient = None  # trigger fallback

# Fallback: use yfinance for historical data if QuoteClient path wasn't usable.
if QuoteClient is None:
    # yfinance often restricts intraday history to recent windows; for long horizons prefer daily bars.
    try:
        # Install if not present
        try:
            import yfinance as yf  # type: ignore
        except Exception:
            # Use %pip so it works in notebooks
            get_ipython().run_line_magic("pip", "install yfinance --quiet")
            import yfinance as yf  # type: ignore

        end_dt = datetime.now(timezone.utc)
        start_dt = end_dt - timedelta(days=365 * years)

        # If user requested intraday (minutes) but years is large, switch to daily to avoid empty results.
        yf_interval = interval
        if interval.endswith("m") and years > 1:
            log.info("Minute-level historicals beyond a few weeks are not available via yfinance; switching to daily.")
            yf_interval = "1d"

        # yfinance accepts start/end as date strings
        df_yf = yf.download(
            tickers=symbol,
            start=start_dt.strftime("%Y-%m-%d"),
            end=(end_dt + timedelta(days=1)).strftime("%Y-%m-%d"),
            interval=yf_interval,
            progress=False,
            threads=False,
        )

        if df_yf.empty:
            raise RuntimeError("yfinance returned no data for the requested range/interval.")

        # Normalize to expected schema
        df_yf = df_yf.rename(
            columns={
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close",
                "Adj Close": "adj_close",
                "Volume": "volume",
            }
        )
        df_yf = df_yf.reset_index()
        # yfinance index for intraday is timezone-aware; ensure UTC
        if isinstance(df_yf.iloc[0, 0], pd.Timestamp):
            df_yf["datetime"] = pd.to_datetime(df_yf.iloc[:, 0], utc=True)
            df_final = df_yf[["datetime", "open", "high", "low", "close", "volume"]].copy()
        else:
            df_final = df_yf.rename(columns={df_yf.columns[0]: "datetime"})
            df_final["datetime"] = pd.to_datetime(df_final["datetime"], utc=True)
            df_final = df_final[["datetime", "open", "high", "low", "close", "volume"]].copy()

        df = df_final
        df.to_csv("spy_5min_10y_webull_fallback_yfinance.csv", index=False)
        display(df.head()), len(df)

    except Exception as e:
        # last resort: surface a clear error
        raise RuntimeError("Failed to fetch data via webull SDK and yfinance fallback: " + str(e))
else:
    # Normalize rows collected from webull SDK into dataframe
    df = pd.DataFrame(all_rows).drop_duplicates().reset_index(drop=True)

    # Try to find/convert a timestamp field if present
    if "timestamp" in df.columns:
        df["datetime"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    elif "time" in df.columns:
        df["datetime"] = pd.to_datetime(df["time"], unit="ms", utc=True)
    elif "ts" in df.columns:
        df["datetime"] = pd.to_datetime(df["ts"], unit="ms", utc=True)
    else:
        # if no explicit timestamp, try to parse any date-like column
        date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
        if date_cols:
            df["datetime"] = pd.to_datetime(df[date_cols[0]], utc=True)
        else:
            raise RuntimeError("No recognizable timestamp field in returned bars.")

    df = df.sort_values("datetime")
    # Keep a standard OHLCV schema if fields exist
    preferred_cols = ["datetime", "open", "high", "low", "close", "volume"]
    df = df[[c for c in preferred_cols if c in df.columns]]
    df.to_csv("spy_5min_10y_webull.csv", index=False)
    display(df.head()), len(df)

/var/folders/g7/ncym07g931q204j4fs767gkh0000gn/T/ipykernel_43720/1620318409.py:131: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_yf = yf.download(


Price,datetime,open,high,low,close,volume
Ticker,,SPY,SPY,SPY,SPY,SPY
0,2016-05-16 00:00:00+00:00,174.131870,176.153884,174.072392,175.678116,77486800
1,2016-05-17 00:00:00+00:00,175.406246,175.695103,173.511652,174.038406,114924900
2,2016-05-18 00:00:00+00:00,173.690047,175.270284,173.001882,174.089355,120062100
3,2016-05-19 00:00:00+00:00,173.367218,173.775018,172.279745,173.486160,115430500
4,2016-05-20 00:00:00+00:00,174.097881,175.100403,174.046908,174.582153,104990400
